# Standardize geographic identifiers

## Step 1A — Standardize Geographic Identifiers

### Purpose

The BEA and BLS datasets use five-digit geographic area codes, while
the project needs a consistent two-digit state FIPS code.

This step cleans the geographic codes, removes national totals and
Washington, D.C., and retains the 50 U.S. states.

No economic observations or missing values are modified.

In [ ]:
from pathlib import Path
import pandas as pd

state_reference = Path("data") / "raw"

In [ ]:
# ============================================================
# 1. STATE REFERENCE - Define the 50 States
# ============================================================

STATE_FIPS = {
    "01": "Alabama", "02": "Alaska", "04": "Arizona",
    "05": "Arkansas", "06": "California", "08": "Colorado",
    "09": "Connecticut", "10": "Delaware", "12": "Florida",
    "13": "Georgia", "15": "Hawaii", "16": "Idaho",
    "17": "Illinois", "18": "Indiana", "19": "Iowa",
    "20": "Kansas", "21": "Kentucky", "22": "Louisiana",
    "23": "Maine", "24": "Maryland", "25": "Massachusetts",
    "26": "Michigan", "27": "Minnesota", "28": "Mississippi",
    "29": "Missouri", "30": "Montana", "31": "Nebraska",
    "32": "Nevada", "33": "New Hampshire", "34": "New Jersey",
    "35": "New Mexico", "36": "New York",
    "37": "North Carolina", "38": "North Dakota",
    "39": "Ohio", "40": "Oklahoma", "41": "Oregon",
    "42": "Pennsylvania", "44": "Rhode Island",
    "45": "South Carolina", "46": "South Dakota",
    "47": "Tennessee", "48": "Texas", "49": "Utah",
    "50": "Vermont", "51": "Virginia", "53": "Washington",
    "54": "West Virginia", "55": "Wisconsin",
    "56": "Wyoming"
}

state_reference = pd.DataFrame({
    "state_fips": list(STATE_FIPS.keys()),
    "state": list(STATE_FIPS.values())
})

print("States in reference table:", len(state_reference))
display(state_reference.head())

In [ ]:
BEA_DATA_DIR = Path("data") / "raw" / "bea"

gdp = pd.read_csv(
    BEA_DATA_DIR / "bea_SAGDP9_raw.csv",
    dtype={"GeoFIPS": "string"},
    low_memory=False
)

income = pd.read_csv(
    BEA_DATA_DIR / "bea_SAINC1_raw.csv",
    dtype={"GeoFIPS": "string"},
    low_memory=False
)

economic_profile = pd.read_csv(
    BEA_DATA_DIR / "bea_SAINC30_raw.csv",
    dtype={"GeoFIPS": "string"},
    low_memory=False
)

print("gdp:", gdp.shape)
print("income:", income.shape)
print("economic_profile:", economic_profile.shape)

In [ ]:
DATA_DIR = Path("data") / "raw" 
industry= pd.read_csv(DATA_DIR / "bls_qcew_industry_employment_2015_2025_raw.csv")
print(industry.shape)

In [ ]:
# ============================================================
# 2. GEOGRAPHIC IDENTIFIER CLEANING FUNCTION - Create a geographic-code cleaning function
# ============================================================

def extract_state_fips(series):
    """
    Convert a BEA or QCEW statewide area code such as
    '"01000"' or '01000' into the two-digit state FIPS '01'.

    Non-state geographic codes return missing values.
    """

    cleaned_codes = (
        series
        .astype("string")
        .str.replace('"', "", regex=False)
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
        .str.zfill(5)
    )

    state_fips = cleaned_codes.str.extract(
        r"^(\d{2})000$",
        expand=False
    )

    return state_fips

In [ ]:

# ============================================================
# 3. Create a resusable state-filtering function
# ============================================================

def retain_50_states(dataframe, geographic_column):
    """
    Standardize geographic identifiers and retain only
    the 50 U.S. states.
    """

    cleaned_df = dataframe.copy()

    cleaned_df["state_fips"] = extract_state_fips(
        cleaned_df[geographic_column]
    )

    cleaned_df = cleaned_df.loc[
        cleaned_df["state_fips"].isin(STATE_FIPS)
    ].copy()

    cleaned_df["state"] = cleaned_df[
        "state_fips"
    ].map(STATE_FIPS)

    return cleaned_df

## Step 1A-1 — Clean BEA identifiers

In [ ]:
# ============================================================
# RETAIN 50 STATES IN BEA TABLES
# ============================================================

sagdp9_states = retain_50_states(
    gdp,
    geographic_column="GeoFIPS"
)

sainc1_states = retain_50_states(
    income,
    geographic_column="GeoFIPS"
)

sainc30_states = retain_50_states(
    economic_profile,
    geographic_column="GeoFIPS"
)

In [ ]:

# ============================================================
# Verify BEA state coverage
# ============================================================

bea_state_check = pd.DataFrame([
    {
        "Dataset": "SAGDP9",
        "Rows": len(sagdp9_states),
        "Unique_States": sagdp9_states["state"].nunique()
    },
    {
        "Dataset": "SAINC1",
        "Rows": len(sainc1_states),
        "Unique_States": sainc1_states["state"].nunique()
    },
    {
        "Dataset": "SAINC30",
        "Rows": len(sainc30_states),
        "Unique_States": sainc30_states["state"].nunique()
    }
])

display(bea_state_check)





## Step 1A-2 — Clean BLS identifiers

In [ ]:
# ============================================================
# RETAIN 50 STATES IN BLS DATA
# ============================================================

bls_industry_states = retain_50_states(
    industry,
    geographic_column="area_fips"
)

bls_industry_states["year"] = pd.to_numeric(
    bls_industry_states["year"],
    errors="coerce"
).astype("Int64")

In [ ]:
# ============================================================
# Check state-year coverage
# ============================================================

bls_state_check = (
    bls_industry_states
    .groupby("year")
    .agg(
        Rows=("state_fips", "size"),
        Unique_States=("state_fips", "nunique")
    )
    .reset_index()
)

display(bls_state_check)

In [ ]:
print("BLS state-year rows:", len(bls_industry_states))

## Step 1A-3 — Confirm Illinois

In [ ]:
# ============================================================
# CONFIRM REFERENCE STATE
# ============================================================

illinois_check = (
    bls_industry_states
    .loc[
        bls_industry_states["state"] == "Illinois",
        ["state_fips", "state", "year"]
    ]
    .sort_values("year")
)

display(illinois_check)

## Step 1A-4 — Check duplicate identifiers

In [ ]:
bls_duplicate_count = (
    bls_industry_states
    .duplicated(
        subset=["state_fips", "year"]
    )
    .sum()
)

print(
    "Duplicate BLS state-year rows:",
    bls_duplicate_count
)

## Step 1A-5 — Summary

In [ ]:
geographic_cleaning_summary = pd.DataFrame([
    {
        "Dataset": "SAGDP9",
        "Rows_Retained": len(sagdp9_states),
        "States_Retained": sagdp9_states["state"].nunique()
    },
    {
        "Dataset": "SAINC1",
        "Rows_Retained": len(sainc1_states),
        "States_Retained": sainc1_states["state"].nunique()
    },
    {
        "Dataset": "SAINC30",
        "Rows_Retained": len(sainc30_states),
        "States_Retained": sainc30_states["state"].nunique()
    },
    {
        "Dataset": "BLS Industry",
        "Rows_Retained": len(bls_industry_states),
        "States_Retained": bls_industry_states["state"].nunique()
    }
])

display(geographic_cleaning_summary)

## Step 1A Interpretation

The geographic identifiers were standardized across the BEA and BLS
datasets.

National totals, Washington, D.C., and other non-state geographic areas
were excluded. All four datasets now contain the same 50-state
population.

The BLS dataset contains one row per state-year. The BEA datasets still
contain multiple rows per state because they include different line
codes and economic measures.

No missing values were removed or imputed. The next step will select the
required BEA measures and reshape the year columns into long
state-year format.

# Select and reshape BEA measures

## Step 1B — Select and Reshape BEA Measures

### Purpose

The BEA files contain many economic measures identified by line codes.
This step identifies the required measures, selects them, and reshapes
the annual columns into a consistent state-year format.

Only verified BEA descriptions and line codes will be used. This avoids
selecting a variable based only on its position in the dataset.

The primary common modeling period is 2015–2024 because SAINC30 ends in
2024. The available 2025 observations will remain preserved in the raw
files for future updates.

In [ ]:
# ============================================================
# 1B-1 STANDARDIZE BEA LINE CODES
# ============================================================

for dataframe in [
    sagdp9_states,
    sainc1_states,
    sainc30_states
]:
    dataframe["LineCode"] = pd.to_numeric(
        dataframe["LineCode"],
        errors="coerce"
    ).astype("Int64")

In [ ]:
# ============================================================
# 1B-2 CREATE LINE-CODE CATALOGS
# ============================================================

def create_bea_catalog(dataframe):
    catalog_columns = [
        "TableName",
        "LineCode",
        "IndustryClassification",
        "Description",
        "Unit"
    ]

    available_columns = [
        column for column in catalog_columns
        if column in dataframe.columns
    ]

    return (
        dataframe[available_columns]
        .drop_duplicates()
        .sort_values("LineCode")
        .reset_index(drop=True)
    )


sagdp9_catalog = create_bea_catalog(sagdp9_states)
sainc1_catalog = create_bea_catalog(sainc1_states)
sainc30_catalog = create_bea_catalog(sainc30_states)

In [ ]:
print("SAGDP9 measures:")
display(sagdp9_catalog)

print("SAINC1 measures:")
display(sainc1_catalog)

In [ ]:
# ============================================================
# Step 1B-3 — Search SAINC30 for relevant measures
# ============================================================

sainc30_keywords = (
    "employment|jobs|earnings|wages|salary|"
    "compensation|population|per capita"
)

sainc30_candidates = sainc30_catalog.loc[
    sainc30_catalog["Description"]
    .str.contains(
        sainc30_keywords,
        case=False,
        na=False,
        regex=True
    )
].copy()

display(sainc30_candidates)

In [ ]:
display(sainc30_catalog)

### Interpretation

SAGDP9 line code 1 measures total real GDP across all industries.

SAINC1 line code 2 measures population, and line code 3 measures
per-capita personal income.

SAINC30 contains multiple employment and earnings concepts. These
measures should be selected according to their complete descriptions
and units rather than by assuming that all employment-related variables
measure the same concept.

In [ ]:
# ============================================================
# Step 1B-4 — Define the common period
# ============================================================

MODEL_START_YEAR = 2015
MODEL_END_YEAR = 2024

YEAR_COLUMNS = [
    str(year)
    for year in range(
        MODEL_START_YEAR,
        MODEL_END_YEAR + 1
    )
]

print("Selected years:", YEAR_COLUMNS)

In [ ]:
for dataset_name, dataframe in {
    "SAGDP9": sagdp9_states,
    "SAINC1": sainc1_states,
    "SAINC30": sainc30_states
}.items():

    missing_years = [
        year for year in YEAR_COLUMNS
        if year not in dataframe.columns
    ]

    print(
        dataset_name,
        "missing selected years:",
        missing_years
    )

In [ ]:
# ============================================================
# Step 1B-5 — Create a BEA reshaping function
# ============================================================

def select_bea_measure(
    dataframe,
    line_code,
    feature_name,
    year_columns=YEAR_COLUMNS
):
    """
    Select one BEA line code and reshape its annual columns
    into one state-year observation per row.
    """

    selected = dataframe.loc[
        dataframe["LineCode"].eq(line_code),
        ["state_fips", "state"] + year_columns
    ].copy()

    if selected.empty:
        raise ValueError(
            f"Line code {line_code} was not found "
            f"for {feature_name}."
        )

    if selected["state_fips"].nunique() != 50:
        raise ValueError(
            f"{feature_name} contains "
            f"{selected['state_fips'].nunique()} states, "
            f"not 50."
        )

    long_data = selected.melt(
        id_vars=["state_fips", "state"],
        value_vars=year_columns,
        var_name="year",
        value_name=feature_name
    )

    long_data["year"] = pd.to_numeric(
        long_data["year"],
        errors="raise"
    ).astype(int)

    long_data[feature_name] = pd.to_numeric(
        long_data[feature_name]
        .astype("string")
        .str.replace(",", "", regex=False),
        errors="coerce"
    )

    long_data = (
        long_data
        .sort_values(["state_fips", "year"])
        .reset_index(drop=True)
    )

    return long_data

In [ ]:
# ============================================================
# Step 1B-6 — Select the verified core measures
# ============================================================

In [ ]:
## Total real GDP

real_gdp = select_bea_measure(
    dataframe=sagdp9_states,
    line_code=1,
    feature_name="real_gdp_millions"
)

In [ ]:
## Population 

population = select_bea_measure(
    dataframe=sainc1_states,
    line_code=2,
    feature_name="population"
)

In [ ]:
## Per-capita personal income 

personal_income_per_capita = select_bea_measure(
    dataframe=sainc1_states,
    line_code=3,
    feature_name="personal_income_per_capita"
)

In [ ]:
# ============================================================
# Step 1B-7 — Validate the reshaped outputs
# ============================================================


core_bea_summary = pd.DataFrame([
    {
        "Feature": "Real GDP",
        "Rows": len(real_gdp),
        "States": real_gdp["state"].nunique(),
        "First_Year": real_gdp["year"].min(),
        "Last_Year": real_gdp["year"].max(),
        "Missing": real_gdp[
            "real_gdp_millions"
        ].isna().sum()
    },
    {
        "Feature": "Population",
        "Rows": len(population),
        "States": population["state"].nunique(),
        "First_Year": population["year"].min(),
        "Last_Year": population["year"].max(),
        "Missing": population[
            "population"
        ].isna().sum()
    },
    {
        "Feature": "Personal income per capita",
        "Rows": len(personal_income_per_capita),
        "States": personal_income_per_capita[
            "state"
        ].nunique(),
        "First_Year": personal_income_per_capita[
            "year"
        ].min(),
        "Last_Year": personal_income_per_capita[
            "year"
        ].max(),
        "Missing": personal_income_per_capita[
            "personal_income_per_capita"
        ].isna().sum()
    }
])

display(core_bea_summary)

In [ ]:
# ============================================================
# Step 1B-8 — Merge the core BEA measures
# ============================================================

bea_core = (
    real_gdp
    .merge(
        population,
        on=["state_fips", "state", "year"],
        how="outer",
        validate="one_to_one"
    )
    .merge(
        personal_income_per_capita,
        on=["state_fips", "state", "year"],
        how="outer",
        validate="one_to_one"
    )
)

bea_core = (
    bea_core
    .sort_values(["state_fips", "year"])
    .reset_index(drop=True)
)

print("BEA core shape:", bea_core.shape)
display(bea_core.head())

In [ ]:
# ============================================================
# Step 1B-9 — Calculate real GDP per capita
# ============================================================

## SAGDP9 is expressed in millions of chained dollars, so multiply by one million before dividing by population:
 
bea_core["real_gdp_per_capita"] = (
    bea_core["real_gdp_millions"]
    * 1_000_000
    / bea_core["population"]
)

## Validata: 

display(
    bea_core[
        [
            "state",
            "year",
            "real_gdp_millions",
            "population",
            "real_gdp_per_capita"
        ]
    ].head()
)

In [ ]:
bea_core[
    [
        "real_gdp_per_capita",
        "personal_income_per_capita"
    ]
].describe()

In [ ]:
# ============================================================
# Step 1B-10 — Inspect Illinois
# ============================================================

illinois_bea_check = bea_core.loc[
    bea_core["state"].eq("Illinois"),
    [
        "state",
        "year",
        "real_gdp_per_capita",
        "personal_income_per_capita"
    ]
]

display(illinois_bea_check)



## Step 1B Interpretation

The required BEA line codes were identified and reshaped from wide annual
tables into state-year format.

The resulting core dataset contains 500 observations representing
**50 states across 10 years from 2015 through 2024**.

Real GDP per capita was calculated by converting real GDP from millions
of chained dollars into dollars and dividing it by state population.

Per-capita personal income remains a nominal measure at this stage. It
should not be described as real income unless an appropriate regional
price or inflation adjustment is applied.

SAINC30 employment and earnings measures have not yet been selected.
Their descriptions and units must first be reviewed to ensure that the
chosen features match the intended economic concepts.

## Save the completed BEA core 

In [ ]:
from pathlib import Path

INTERIM_DIR = Path("data") / "interim"
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

bea_core.to_csv(
    INTERIM_DIR / "bea_core_2015_2024.csv",
    index=False
)

print(
    "Saved:",
    INTERIM_DIR / "bea_core_2015_2024.csv"
)

## data preparation 

- data/processed/state_economic_annual_prepared.csv
- data/processed/baseline_structural.csv
- data/processed/shock_structural.csv
- data/processed/post_shock_structural.csv
- data/processed/feature_manifest.csv

In [ ]:
display(sainc30_candidates)

The SAINC30 output gives us two appropriate measures:  

| Line code | Measure                         | Use                           |
| --------: | ------------------------------- | ----------------------------- |
|       240 | Total employment—number of jobs | Calculate real output per job |
|       300 | Average wages and salaries      | Wage-level feature            |


# Select employment and wage measures

## Step 1C — Employment, Wages, and Output per Job

### Purpose

This step selects total employment and average wages from SAINC30.

Total employment is used with real GDP to calculate real output per job,
which serves as a state-level labor-productivity proxy.

Average wages and salaries represent the average wage level associated
with wage-and-salary employment.

These employment measures count jobs rather than unique workers. A
person with multiple jobs may therefore be counted more than once.

In [ ]:
# ============================================================
# TOTAL EMPLOYMENT — SAINC30 LINE 240
# ============================================================

total_employment = select_bea_measure(
    dataframe=sainc30_states,
    line_code=240,
    feature_name="total_employment_jobs"
)

In [ ]:
# ============================================================
# AVERAGE WAGES AND SALARIES — SAINC30 LINE 300
# ============================================================

average_wages = select_bea_measure(
    dataframe=sainc30_states,
    line_code=300,
    feature_name="average_wages_salaries"
)

In [ ]:
## Validate the selected measures

sainc30_feature_summary = pd.DataFrame([
    {
        "Feature": "Total employment",
        "LineCode": 240,
        "Rows": len(total_employment),
        "States": total_employment["state"].nunique(),
        "First_Year": total_employment["year"].min(),
        "Last_Year": total_employment["year"].max(),
        "Missing": total_employment[
            "total_employment_jobs"
        ].isna().sum()
    },
    {
        "Feature": "Average wages and salaries",
        "LineCode": 300,
        "Rows": len(average_wages),
        "States": average_wages["state"].nunique(),
        "First_Year": average_wages["year"].min(),
        "Last_Year": average_wages["year"].max(),
        "Missing": average_wages[
            "average_wages_salaries"
        ].isna().sum()
    }
])

display(sainc30_feature_summary)


In [ ]:
# ============================================================
# MERGE SAINC30 MEASURES WITH BEA CORE
# ============================================================

bea_economic = (
    bea_core
    .merge(
        total_employment,
        on=["state_fips", "state", "year"],
        how="left",
        validate="one_to_one"
    )
    .merge(
        average_wages,
        on=["state_fips", "state", "year"],
        how="left",
        validate="one_to_one"
    )
)

bea_economic = (
    bea_economic
    .sort_values(["state_fips", "year"])
    .reset_index(drop=True)
)

print("BEA economic dataset:", bea_economic.shape)
display(bea_economic.head())

In [ ]:
assert "real_gdp_per_capita" in bea_economic.columns

In [ ]:
# ============================================================
# Calculate REAL OUTPUT PER JOB
# ============================================================

bea_economic["real_output_per_job"] = (
    bea_economic["real_gdp_millions"]
    * 1_000_000
    / bea_economic["total_employment_jobs"]
)

Use the name real output per job or labor-productivity proxy.\
It is not a perfect labor-productivity measure because:\
- The denominator is jobs, not hours worked.
- Part-time and full-time jobs receive the same job count.
- Workers with multiple jobs may be counted more than once.

In [ ]:
## Review the calculated features 

economic_feature_summary = bea_economic[
    [
        "real_gdp_per_capita",
        "personal_income_per_capita",
        "total_employment_jobs",
        "average_wages_salaries",
        "real_output_per_job"
    ]
].describe().T

display(economic_feature_summary)

In [ ]:
## Check invalid values 

import numpy as np 

quality_check = pd.DataFrame([
    {
        "Feature": feature,
        "Missing": bea_economic[feature].isna().sum(),
        "Infinite": np.isinf(
            bea_economic[feature]
        ).sum(),
        "Zero_or_Negative": (
            bea_economic[feature] <= 0
        ).sum()
    }
    for feature in [
        "real_gdp_per_capita",
        "personal_income_per_capita",
        "total_employment_jobs",
        "average_wages_salaries",
        "real_output_per_job"
    ]
])

display(quality_check)

In [ ]:
# ============================================================
# Inspect Illinois 
# ============================================================

illinois_economic_check = bea_economic.loc[
    bea_economic["state"].eq("Illinois"),
    [
        "state",
        "year",
        "total_employment_jobs",
        "average_wages_salaries",
        "real_output_per_job"
    ]
]

display(illinois_economic_check)



In [ ]:
# ============================================================
# Save an undated checkpoint
# ============================================================

from pathlib import Path

INTERIM_DIR = Path("data") / "interim"
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

bea_economic.to_csv(
    INTERIM_DIR / "bea_economic_2015_2024.csv",
    index=False
)

print(
    "Saved:",
    INTERIM_DIR / "bea_economic_2015_2024.csv"
)

## Step 1C Interpretation

Total employment and average wages were selected from SAINC30 using
verified line codes 240 and 300.

Total employment was combined with real GDP to calculate real output per
job. This measure serves as a labor-productivity proxy, but it does not
account for differences in hours worked or multiple-job holders.

Average wages and personal income are currently nominal dollar measures.
Before comparing economic positions across time, these features should
be converted into inflation-adjusted values. Otherwise, general price
growth could appear as economic movement even when the relative
structure of states did not change.

The updated BEA dataset contains one observation for every state-year
from 2015 through 2024.

# Calculate industry-employment shares

## Step 1D — Prepare BLS Industry Employment Shares

### Purpose

Raw industry-employment counts cannot be compared directly across states
because large states naturally employ more people.

This step converts selected industries into percentages of each state's
total QCEW employment.

The project uses QCEW total employment as the denominator so that the
numerator and denominator come from the same data source and employment
coverage definition.

In [ ]:
from pathlib import Path

import pandas as pd
import time

In [ ]:
qcew_total_raw = pd.read_csv(
    "data/raw/bls/"
    "bls_qcew_total_employment_2015_2025_raw.csv",
    dtype={"area_fips": "string"}
)

print(qcew_total_raw.shape)
display(qcew_total_raw.head())

### Interpretation

The 20 industry datasets had already been collected from the BLS QCEW
program. This additional collection retrieves total QCEW employment for
each state-year.

Total employment will be used as the denominator when converting
industry-employment counts into shares.

The earlier NameError occurred because the original collection function
was not available in the new notebook. It did not indicate that the BLS
data were unavailable.

In [ ]:
# ============================================================
# RETAIN THE 50 STATES
# ============================================================

qcew_total_states = retain_50_states(
    qcew_total_raw,
    geographic_column="area_fips"
)

qcew_total_states["year"] = pd.to_numeric(
    qcew_total_states["year"],
    errors="raise"
).astype(int)

qcew_total_states[
    "QCEW_Total_Employment"
] = pd.to_numeric(
    qcew_total_states["QCEW_Total_Employment"],
    errors="coerce"
)

qcew_total_states = (
    qcew_total_states
    .sort_values(["year", "state_fips"])
    .reset_index(drop=True)
)

print("Filtered shape:", qcew_total_states.shape)
print(
    "States:",
    qcew_total_states["state"].nunique()
)
print(
    "Years:",
    qcew_total_states["year"].min(),
    "to",
    qcew_total_states["year"].max()
)

In [ ]:
## Validate annual coverage 

total_employment_coverage = (
    qcew_total_states
    .groupby("year")
    .agg(
        Rows=("state_fips", "size"),
        States=("state_fips", "nunique"),
        Missing=(
            "QCEW_Total_Employment",
            lambda values: values.isna().sum()
        )
    )
    .reset_index()
)

display(total_employment_coverage)

In [ ]:
duplicate_count = (
    qcew_total_states
    .duplicated(["state_fips", "year"])
    .sum()
)

print(
    "Duplicate state-year rows:",
    duplicate_count
)

In [ ]:
INTERIM_DIR = Path("data") / "interim"
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

qcew_total_states.to_csv(
    INTERIM_DIR
    / "qcew_total_employment_50_states_2015_2025.csv",
    index=False
)

print("Filtered QCEW total-employment data saved.")

In [ ]:
## Select the primary 2015–2024 period

qcew_total_model = (
    qcew_total_states
    .loc[
        qcew_total_states["year"].between(
            2015,
            2024
        ),
        [
            "state_fips",
            "state",
            "year",
            "QCEW_Total_Employment"
        ]
    ]
    .copy()
)

print("Modeling shape:", qcew_total_model.shape)


### Interpretation

QCEW total employment was successfully collected for 53 geographic
areas from 2015 through 2025.

After excluding Washington, D.C., Puerto Rico, and the U.S. Virgin
Islands, the dataset contains 550 observations representing 50 states
over 11 years.

The earlier failure messages came from an outdated code cell that called
a function unavailable in the current notebook. They do not indicate a
failure in the saved total-employment dataset.

In [ ]:
# ============================================================
# Step 1D-4 Prepare BLS industry data for 2015-2024 
# ============================================================

industry_model = (
    bls_industry_states
    .loc[
        bls_industry_states["year"].between(
            2015,
            2024
        )
    ]
    .copy()
)

qcew_total_model = (
    qcew_total_states
    .loc[
        qcew_total_states["year"].between(
            2015,
            2024
        ),
        [
            "state_fips",
            "state",
            "year",
            "QCEW_Total_Employment"
        ]
    ]
    .copy()
)

print("Industry rows:", len(industry_model))
print("Total-employment rows:", len(qcew_total_model))

In [ ]:
print("Industry model shape:", industry_model.shape)
print(industry_model.columns.tolist())

In [ ]:
qcew_total_model.columns

In [ ]:
# ============================================================
# Detect the industry columns automatically
# ============================================================

# Columns that are identifiers, not industries
IDENTIFIER_COLUMNS = {
    "area_fips",
    "state_fips",
    "state",
    "year"
}

INDUSTRY_COLUMNS = [
    column
    for column in industry_model.columns
    if column not in IDENTIFIER_COLUMNS
]

print(
    "Number of industry columns:",
    len(INDUSTRY_COLUMNS)
)

print(INDUSTRY_COLUMNS)

In [ ]:
# ============================================================
# Validate that 20 industries exist
# ============================================================

if len(INDUSTRY_COLUMNS) != 20:
    raise ValueError(
        "Expected 20 industry columns, but found "
        f"{len(INDUSTRY_COLUMNS)}.\n"
        f"Detected columns: {INDUSTRY_COLUMNS}"
    )

In [ ]:
# ============================================================
# Convert industry employment to numeric
# ============================================================

industry_model[
    INDUSTRY_COLUMNS
] = industry_model[
    INDUSTRY_COLUMNS
].apply(
    pd.to_numeric,
    errors="coerce"
)

In [ ]:
# ============================================================
# Check missing industry values
# ============================================================

industry_missingness = (
    industry_model[
        INDUSTRY_COLUMNS
    ]
    .isna()
    .sum()
    .sort_values(ascending=False)
    .rename("Missing")
    .to_frame()
)

industry_missingness["Coverage_Percent"] = (
    100
    * (
        len(industry_model)
        - industry_missingness["Missing"]
    )
    / len(industry_model)
)

display(
    industry_missingness.loc[
        industry_missingness["Missing"] > 0
    ]
)

## Merge industries with total employment

In [ ]:
## ensure both FIPS columns are strings: 

industry_model["state_fips"] = (
    industry_model["state_fips"]
    .astype("string")
    .str.zfill(2)
)

qcew_total_model["state_fips"] = (
    qcew_total_model["state_fips"]
    .astype("string")
    .str.zfill(2)
)

In [ ]:
# ============================================================
# MERGE INDUSTRIES WITH TOTAL EMPLOYMENT
# ============================================================

bls_industry_model = industry_model.merge(
    qcew_total_model,
    on=["state_fips", "state", "year"],
    how="left",
    validate="one_to_one"
)

print(
    "Merged BLS industry shape:",
    bls_industry_model.shape
)

display(bls_industry_model.head())

In [ ]:
## Confirm the denominator was matched

print(
    "Missing QCEW total employment:",
    bls_industry_model[
        "QCEW_Total_Employment"
    ].isna().sum()
)

In [ ]:
# ============================================================
# Construct the industry groups
# ============================================================

# Natural resources and agriculture
bls_industry_model[
    "Natural_Resources_Agriculture"
] = bls_industry_model[
    ["Agriculture", "Mining"]
].sum(
    axis=1,
    min_count=2
)

# Professional and business services
bls_industry_model[
    "Professional_Business_Services"
] = bls_industry_model[
    [
        "Professional_Technical",
        "Management_of_Companies",
        "Administrative_Support"
    ]
].sum(
    axis=1,
    min_count=3
)

In [ ]:
# ============================================================
# Calculate the four industry shares
# ============================================================

share_definitions = {
    "manufacturing_share":
        "Manufacturing",

    "professional_business_share":
        "Professional_Business_Services",

    "healthcare_share":
        "Health_Care",

    "natural_resources_agriculture_share":
        "Natural_Resources_Agriculture"
}

for share_name, employment_column in (
    share_definitions.items()
):
    bls_industry_model[share_name] = (
        100
        * bls_industry_model[employment_column]
        / bls_industry_model[
            "QCEW_Total_Employment"
        ]
    )

In [ ]:
## Validate the shares

SHARE_COLUMNS = list(
    share_definitions.keys()
)

share_quality = pd.DataFrame([
    {
        "Feature": feature,
        "Missing": bls_industry_model[
            feature
        ].isna().sum(),
        "Minimum": bls_industry_model[
            feature
        ].min(),
        "Maximum": bls_industry_model[
            feature
        ].max(),
        "Below_Zero": (
            bls_industry_model[feature] < 0
        ).sum(),
        "Above_100": (
            bls_industry_model[feature] > 100
        ).sum()
    }
    for feature in SHARE_COLUMNS
])

display(share_quality)

- Clarification 


| Object               | Contents                                                |
| -------------------- | ------------------------------------------------------- |
| `industry_model`     | The 20 individual industry-employment columns           |
| `qcew_total_model`   | Total employment denominator only                       |
| `bls_industry_model` | Combined industries, denominator, and calculated shares |
| `QCEW_INDUSTRIES`    | Earlier collection dictionary; no longer required       |


### Interpretation

The industry-employment and total-employment datasets serve different
purposes.

The industry dataset contains employment counts for 20 individual
industries. The total-employment dataset contains only the denominator
used to calculate industry shares.

After merging them by state and year, the model can calculate comparable
industry-employment percentages for every state.

# Merge BEA and BLS features

## Step 1E — Merge BEA and BLS Economic Features

### Purpose

This step combines the BEA economic features with the BLS industry-
employment shares.

The datasets are merged using state FIPS code and year. A one-to-one
validation ensures that every state-year appears only once in each
dataset.

After merging, the combined dataset will be checked for unmatched rows,
missing values, duplicate observations, and incomplete feature coverage.

In [ ]:
# ============================================================
# 1. SELECT BLS INDUSTRY-SHARE FEATURES
# ============================================================

SHARE_COLUMNS = [
    "manufacturing_share",
    "professional_business_share",
    "healthcare_share",
    "natural_resources_agriculture_share"
]

bls_shares = bls_industry_model[
    [
        "state_fips",
        "state",
        "year",
        "QCEW_Total_Employment"
    ]
    + SHARE_COLUMNS
].copy()

bls_shares = bls_shares.rename(
    columns={"state": "state_bls"}
)

print("BLS share dataset:", bls_shares.shape)
display(bls_shares.head())

In [ ]:
# ============================================================
# 2. STANDARDIZE MERGE IDENTIFIERS
# ============================================================

bea_economic["state_fips"] = (
    bea_economic["state_fips"]
    .astype("string")
    .str.zfill(2)
)

bls_shares["state_fips"] = (
    bls_shares["state_fips"]
    .astype("string")
    .str.zfill(2)
)

bea_economic["year"] = pd.to_numeric(
    bea_economic["year"],
    errors="raise"
).astype(int)

bls_shares["year"] = pd.to_numeric(
    bls_shares["year"],
    errors="raise"
).astype(int)

In [ ]:
# ============================================================
# 3. PRE-MERGE DUPLICATE CHECK
# ============================================================

bea_duplicates = bea_economic.duplicated(
    ["state_fips", "year"]
).sum()

bls_duplicates = bls_shares.duplicated(
    ["state_fips", "year"]
).sum()

print("BEA duplicate state-years:", bea_duplicates)
print("BLS duplicate state-years:", bls_duplicates)

In [ ]:
# ============================================================
# 4. MERGE BEA AND BLS
# ============================================================

state_economic_merged = bea_economic.merge(
    bls_shares,
    on=["state_fips", "year"],
    how="outer",
    validate="one_to_one",
    indicator=True
)

print(
    "Merged dataset shape:",
    state_economic_merged.shape
)

In [ ]:
# ============================================================
# 5. Check merge completeness
# ============================================================

merge_summary = (
    state_economic_merged["_merge"]
    .value_counts(dropna=False)
    .rename_axis("Merge_Status")
    .reset_index(name="Rows")
)

display(merge_summary)

In [ ]:
# ============================================================
# 6. Confirm state names match
# ============================================================

state_name_mismatches = (
    state_economic_merged["state"]
    != state_economic_merged["state_bls"]
)

print(
    "State-name mismatches:",
    state_name_mismatches.sum()
)

In [ ]:
state_economic_merged = (
    state_economic_merged
    .drop(columns="state_bls")
)

## Step 1E-1 — Validate the combined structure

In [ ]:
combined_structure = pd.DataFrame({
    "Measure": [
        "Rows",
        "Columns",
        "States",
        "Years",
        "First year",
        "Last year",
        "Duplicate state-years"
    ],
    "Result": [
        state_economic_merged.shape[0],
        state_economic_merged.shape[1],
        state_economic_merged[
            "state_fips"
        ].nunique(),
        state_economic_merged[
            "year"
        ].nunique(),
        state_economic_merged[
            "year"
        ].min(),
        state_economic_merged[
            "year"
        ].max(),
        state_economic_merged.duplicated(
            ["state_fips", "year"]
        ).sum()
    ]
})

display(combined_structure)

## Step 1E-2 — Audit feature missingness

In [ ]:
# ============================================================
# FEATURE MISSINGNESS
# ============================================================

CURRENT_FEATURES = [
    "real_gdp_per_capita",
    "personal_income_per_capita",
    "real_output_per_job",
    "average_wages_salaries",
    "manufacturing_share",
    "professional_business_share",
    "healthcare_share",
    "natural_resources_agriculture_share"
]

missing_features = [
    feature for feature in CURRENT_FEATURES
    if feature not in state_economic_merged.columns
]

if missing_features:
    raise KeyError(
        f"Missing expected features: {missing_features}"
    )

feature_quality = pd.DataFrame({
    "Feature": CURRENT_FEATURES,
    "Missing": [
        state_economic_merged[
            feature
        ].isna().sum()
        for feature in CURRENT_FEATURES
    ]
})

feature_quality["Coverage_Percent"] = (
    100
    * (
        len(state_economic_merged)
        - feature_quality["Missing"]
    )
    / len(state_economic_merged)
)

feature_quality["Status"] = pd.cut(
    feature_quality["Coverage_Percent"],
    bins=[-np.inf, 75, 90, 95, np.inf],
    labels=[
        "Very Low",
        "Low",
        "Review",
        "High"
    ],
    right=False
)

display(feature_quality)

## Step 1E-3 — Inspect missing rows

In [ ]:
rows_with_missing_features = (
    state_economic_merged
    .loc[
        state_economic_merged[
            CURRENT_FEATURES
        ].isna().any(axis=1),
        [
            "state_fips",
            "state",
            "year"
        ]
        + CURRENT_FEATURES
    ]
)

print(
    "State-year rows with missing features:",
    len(rows_with_missing_features)
)

display(rows_with_missing_features)

## Step 1E-4 — Inspect numeric ranges

In [ ]:
feature_range_check = (
    state_economic_merged[
        CURRENT_FEATURES
    ]
    .describe()
    .T
)

display(feature_range_check)

In [ ]:
# Inspect zero industry shares

for feature in [
    "manufacturing_share",
    "natural_resources_agriculture_share"
]:
    zero_rows = state_economic_merged.loc[
        state_economic_merged[feature].eq(0),
        ["state", "year", feature]
    ]

    print(f"\n{feature}: {len(zero_rows)} zero rows")
    display(zero_rows)

## Step 1E-5 — Check Illinois

In [ ]:
illinois_combined = (
    state_economic_merged
    .loc[
        state_economic_merged["state"].eq("Illinois"),
        [
            "state",
            "year"
        ]
        + CURRENT_FEATURES
    ]
    .sort_values("year")
)

display(illinois_combined)

## Step 1E-6 — Save the merged checkpoint

In [ ]:
from pathlib import Path

INTERIM_DIR = Path("data") / "interim"
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

merged_checkpoint = (
    INTERIM_DIR
    / "state_economic_merged_2015_2024.csv"
)

state_economic_merged.to_csv(
    merged_checkpoint,
    index=False
)

print("Saved:", merged_checkpoint)

## Step 1E Interpretation

The BEA economic indicators and BLS industry-employment shares were
successfully merged.

The resulting dataset contains 500 state-year observations covering
50 states from 2015 through 2024. All eight current candidate features
have complete coverage.

The industry-share measures make states comparable despite large
differences in population and total employment. Exact zero industry
shares require a final source-data check before modeling.

## Review the underlying employment values

In [ ]:
# ============================================================
# REVIEW SUSPICIOUS ZERO VALUES
# ============================================================

suspicious_industry_rows = (
    bls_industry_model
    .loc[
        bls_industry_model[
            "manufacturing_share"
        ].eq(0)
        |
        bls_industry_model[
            "natural_resources_agriculture_share"
        ].eq(0),
        [
            "state",
            "year",
            "Manufacturing",
            "Agriculture",
            "Mining",
            "QCEW_Total_Employment",
            "manufacturing_share",
            "natural_resources_agriculture_share"
        ]
    ]
    .sort_values(["state", "year"])
)

display(suspicious_industry_rows)

## Mark the values for imputation

In [ ]:
# ============================================================
# CREATE IMPUTATION FLAGS
# ============================================================

state_economic_merged[
    "manufacturing_share_imputed"
] = state_economic_merged[
    "manufacturing_share"
].eq(0)

state_economic_merged[
    "natural_resources_share_imputed"
] = state_economic_merged[
    "natural_resources_agriculture_share"
].eq(0)

In [ ]:
## Convert the suspicious zeros to missing values:

state_economic_merged.loc[
    state_economic_merged[
        "manufacturing_share"
    ].eq(0),
    "manufacturing_share"
] = np.nan

state_economic_merged.loc[
    state_economic_merged[
        "natural_resources_agriculture_share"
    ].eq(0),
    "natural_resources_agriculture_share"
] = np.nan

## Impute within each state

Because industry composition is a slow-moving structural feature, use interpolation between available state observations. At the beginning or end of the series, use the nearest available observation.

In [ ]:
# ============================================================
# WITHIN-STATE INTERPOLATION
# ============================================================

state_economic_merged = (
    state_economic_merged
    .sort_values(["state_fips", "year"])
    .reset_index(drop=True)
)

features_to_impute = [
    "manufacturing_share",
    "natural_resources_agriculture_share"
]

for feature in features_to_impute:
    state_economic_merged[feature] = (
        state_economic_merged
        .groupby("state_fips")[feature]
        .transform(
            lambda values:
                values.interpolate(
                    method="linear",
                    limit_direction="both"
                )
        )
    )

For **Alaska** in 2020, linear interpolation uses the neighboring 2019 and 2021 values.\
For **Delaware** and **Rhode** Island:
- Missing early years use the nearest first valid observation.
- Internal gaps use linear interpolation.
- Missing final years use the nearest last valid observation.

## Validate the result

In [ ]:
imputation_summary = pd.DataFrame([
    {
        "Feature": "Manufacturing share",
        "Values_Imputed": int(
            state_economic_merged[
                "manufacturing_share_imputed"
            ].sum()
        ),
        "Remaining_Missing": int(
            state_economic_merged[
                "manufacturing_share"
            ].isna().sum()
        ),
        "Remaining_Zero": int(
            state_economic_merged[
                "manufacturing_share"
            ].eq(0).sum()
        )
    },
    {
        "Feature": "Natural resources/agriculture share",
        "Values_Imputed": int(
            state_economic_merged[
                "natural_resources_share_imputed"
            ].sum()
        ),
        "Remaining_Missing": int(
            state_economic_merged[
                "natural_resources_agriculture_share"
            ].isna().sum()
        ),
        "Remaining_Zero": int(
            state_economic_merged[
                "natural_resources_agriculture_share"
            ].eq(0).sum()
        )
    }
])

display(imputation_summary)

In [ ]:
# Review the repaired rows

repaired_rows = state_economic_merged.loc[
    state_economic_merged[
        "manufacturing_share_imputed"
    ]
    |
    state_economic_merged[
        "natural_resources_share_imputed"
    ],
    [
        "state",
        "year",
        "manufacturing_share",
        "natural_resources_agriculture_share",
        "manufacturing_share_imputed",
        "natural_resources_share_imputed"
    ]
]

display(repaired_rows)

In [ ]:
# Features repaired during Step 4E
REPAIRED_FEATURES = [
    "manufacturing_share",
    "natural_resources_agriculture_share"
]

repair_validation = pd.DataFrame({
    "Feature": REPAIRED_FEATURES,
    "Imputed_Rows": [
        state_economic_merged["manufacturing_share_imputed"].sum(),
        state_economic_merged["natural_resources_share_imputed"].sum()
    ],
    "Remaining_Missing": [
        state_economic_merged["manufacturing_share"].isna().sum(),
        state_economic_merged[
            "natural_resources_agriculture_share"
        ].isna().sum()
    ],
    "Remaining_Zero": [
        state_economic_merged["manufacturing_share"].eq(0).sum(),
        state_economic_merged[
            "natural_resources_agriculture_share"
        ].eq(0).sum()
    ]
})

display(repair_validation)

assert (
    state_economic_merged[REPAIRED_FEATURES]
    .notna()
    .all()
    .all()
)

assert (
    state_economic_merged[REPAIRED_FEATURES]
    .gt(0)
    .all()
    .all()
)

print("Industry-share repair validation passed.")

### Interpretation of industry-share repairs

The industry-share quality check identified one suspicious zero for
Alaska's manufacturing share and fourteen suspicious zeros for the
natural-resources-and-agriculture share in Delaware and Rhode Island.

Alaska's 2020 manufacturing share was interpolated using the surrounding
years. Because the missing Delaware and Rhode Island observations occurred
at the beginning or end of their time series, their values were filled using
the nearest observed state value.

Imputation flags were retained so that the final clustering results can later
be tested with and without these repaired observations. These repairs affect
15 of 500 state-year observations and do not create a large general missing-data
problem.

## Save the repaired checkpoint

In [ ]:
from pathlib import Path

INTERIM_DIR = Path("data/interim")
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

checkpoint_path = (
    INTERIM_DIR /
    "state_economic_merged_repaired_2015_2024.csv"
)

state_economic_merged.to_csv(
    checkpoint_path,
    index=False
)

print(f"Saved: {checkpoint_path}")
print(f"Shape: {state_economic_merged.shape}")

### Zero-Value Treatment Interpretation

One manufacturing observation and fourteen natural-resource/agriculture
observations were recorded as exact zeros.

The geographic and temporal patterns suggest that these values represent
unavailable or suppressed QCEW observations rather than the complete
absence of industry employment.

Because both features retained more than 95% valid coverage, the
suspicious zeros were converted to missing values and estimated from
each state's available time series.

Imputation flags were retained so the modified observations remain
transparent and can be excluded in a later sensitivity analysis.

## Step 1F — Adjust nominal variables for inflation

We will convert income and wages into constant 2017 dollars because `real_gdp_per_capita` is already measured in chained 2017 dollars.

In [ ]:
# ============================================================
# STEP 4F-1 — ANNUAL CPI-U, 2015–2024 -- Create the CPI table
# Base period: 1982–1984 = 100
# ============================================================

cpi_annual = pd.DataFrame({
    "year": list(range(2015, 2025)),
    "cpi_u": [
        237.017,  # 2015
        240.007,  # 2016
        245.120,  # 2017
        251.107,  # 2018
        255.657,  # 2019
        258.811,  # 2020
        270.970,  # 2021
        292.655,  # 2022
        304.702,  # 2023
        313.689   # 2024
    ]
})

CPI_BASE_YEAR = 2017

cpi_base = cpi_annual.loc[
    cpi_annual["year"].eq(CPI_BASE_YEAR),
    "cpi_u"
].iloc[0]

print(f"CPI base year: {CPI_BASE_YEAR}")
print(f"CPI base value: {cpi_base}")

display(cpi_annual)

### CPI inflation adjustment

The CPI-U measures changes in the average price level paid by urban
consumers in the United States.

The project uses 2017 as the inflation-adjustment base year because the
real GDP variables are reported in chained 2017 dollars. Nominal income
and wage values will therefore be converted into constant 2017 dollars.

## Step 1F-2 — Merge CPI with the economic dataset

In [ ]:
# Make sure year has the same numeric type
state_economic_merged["year"] = pd.to_numeric(
    state_economic_merged["year"],
    errors="raise"
).astype(int)

state_economic_clean = state_economic_merged.merge(
    cpi_annual,
    on="year",
    how="left",
    validate="many_to_one"
)

print("Shape after CPI merge:", state_economic_clean.shape)
print("Missing CPI values:", state_economic_clean["cpi_u"].isna().sum())

## Step 1F-3 — Create inflation-adjusted features

$$
\text{Real value in 2017 dollars}
=
\text{Nominal value}
\times
\frac{\text{CPI}_{2017}}{\text{CPI}_{\text{year}}}
$$

In [ ]:
# ============================================================
# CREATE CONSTANT-2017-DOLLAR FEATURES
# ============================================================

inflation_factor = cpi_base / state_economic_clean["cpi_u"]

state_economic_clean[
    "real_personal_income_per_capita_2017"
] = (
    state_economic_clean["personal_income_per_capita"]
    * inflation_factor
)

state_economic_clean[
    "real_average_wages_salaries_2017"
] = (
    state_economic_clean["average_wages_salaries"]
    * inflation_factor
)

## Step 1F-4 — Validate the adjustment

In [ ]:
real_monetary_features = [
    "real_personal_income_per_capita_2017",
    "real_average_wages_salaries_2017"
]

inflation_quality_check = pd.DataFrame({
    "Feature": real_monetary_features,
    "Missing": [
        state_economic_clean[column].isna().sum()
        for column in real_monetary_features
    ],
    "Infinite": [
        np.isinf(state_economic_clean[column]).sum()
        for column in real_monetary_features
    ],
    "Zero_or_Negative": [
        state_economic_clean[column].le(0).sum()
        for column in real_monetary_features
    ]
})

display(inflation_quality_check)

assert (
    state_economic_clean[real_monetary_features]
    .notna()
    .all()
    .all()
)

assert np.isfinite(
    state_economic_clean[real_monetary_features]
).all().all()

assert (
    state_economic_clean[real_monetary_features]
    .gt(0)
    .all()
    .all()
)

print("Inflation-adjustment validation passed.")

#### Confirm the 2017 base-year calculation
In 2017, nominal and real values should be equal.

In [ ]:
base_year_check = state_economic_clean.loc[
    state_economic_clean["year"].eq(2017),
    [
        "state",
        "personal_income_per_capita",
        "real_personal_income_per_capita_2017",
        "average_wages_salaries",
        "real_average_wages_salaries_2017"
    ]
]

assert np.allclose(
    base_year_check["personal_income_per_capita"],
    base_year_check["real_personal_income_per_capita_2017"]
)

assert np.allclose(
    base_year_check["average_wages_salaries"],
    base_year_check["real_average_wages_salaries_2017"]
)

print("2017 base-year validation passed.")
display(base_year_check.head())

##  Step 1F-5 — Examine Illinois

In [ ]:
illinois_inflation_check = (
    state_economic_clean.loc[
        state_economic_clean["state"].eq("Illinois"),
        [
            "state",
            "year",
            "personal_income_per_capita",
            "real_personal_income_per_capita_2017",
            "average_wages_salaries",
            "real_average_wages_salaries_2017"
        ]
    ]
    .sort_values("year")
)

display(illinois_inflation_check)

### Interpretation of the inflation adjustment

Personal income per capita and average wages were originally measured in
nominal dollars. Nominal values contain both actual economic growth and
general price inflation.

Converting these variables to constant 2017 dollars makes observations
from different years more comparable. This is particularly important
because the analysis compares the baseline, shock, and post-shock periods.

The adjustment uses a national CPI and therefore controls for national
inflation over time. It does not control for differences in the cost of
living between individual states.

## Step 1F-6 — Save the cleaned checkpoint

In [ ]:
from pathlib import Path

INTERIM_DIR = Path("data/interim")
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

cpi_path = INTERIM_DIR / "annual_cpi_u_2015_2024.csv"
clean_checkpoint_path = (
    INTERIM_DIR /
    "state_economic_inflation_adjusted_2015_2024.csv"
)

cpi_annual.to_csv(cpi_path, index=False)

state_economic_clean.to_csv(
    clean_checkpoint_path,
    index=False
)

print(f"Saved CPI table: {cpi_path}")
print(f"Saved cleaned checkpoint: {clean_checkpoint_path}")
print(f"Dataset shape: {state_economic_clean.shape}")

In [ ]:
state_economic_clean.columns.tolist()

In [ ]:
cpi_annual.columns.tolist()

In [ ]:
current_model_features = [
    "real_gdp_per_capita",
    "real_personal_income_per_capita_2017",
    "real_output_per_job",
    "real_average_wages_salaries_2017",
    "manufacturing_share",
    "professional_business_share",
    "healthcare_share",
    "natural_resources_agriculture_share"
]

print("Current structural features:", len(current_model_features))

## Step 1G — Add the four remaining structural features

| Feature                        | Source                    | Treatment of 2020            |
| ------------------------------ | ------------------------- | ---------------------------- |
| Labor-force participation rate | BLS/FRED                  | Use actual 2020 observations |
| Employment-to-population ratio | Calculated from BLS rates | Use actual 2020 observations |
| Bachelor’s-degree attainment   | Census ACS                | Interpolate 2020             |
| Working-age population share   | Census ACS                | Interpolate 2020             |


## Revised Step 1G — Close data preparation

## Model A — State structural similarity

## Model B — Economic change and shock response

Create four growth features from data we already collected:\

- Real GDP per-capita growth
- Real personal-income growth
- Employment growth
- Real wage growth
  
These features will help us interpret how states responded during **2020–2022** without collecting additional datasets.

## The Eight features alrelady cover: 

| Dimension                     | Available features        |
| ----------------------------- | ------------------------- |
| Economic capacity             | GDP and income per capita |
| Productivity                  | Output per job            |
| Labor compensation            | Real wages                |
| Industry employment structure | Four employment shares    |


In [ ]:
# ============================================================
# REVISED STEP 1G-1 — Build the FINAL CORE PANEL
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd


CORE_FEATURES = [
    "real_gdp_per_capita",
    "real_personal_income_per_capita_2017",
    "real_output_per_job",
    "real_average_wages_salaries_2017",
    "manufacturing_share",
    "professional_business_share",
    "healthcare_share",
    "natural_resources_agriculture_share"
]

identifier_columns = [
    "state_fips",
    "state",
    "year"
]

# Preserve available imputation flags
possible_flag_columns = [
    "manufacturing_share_imputed",
    "natural_resources_share_imputed"
]

flag_columns = [
    column
    for column in possible_flag_columns
    if column in state_economic_clean.columns
]

state_structural_panel = (
    state_economic_clean[
        identifier_columns
        + CORE_FEATURES
        + flag_columns
    ]
    .copy()
    .sort_values(["state_fips", "year"])
    .reset_index(drop=True)
)

print("Final panel shape:", state_structural_panel.shape)
display(state_structural_panel.head())

In [ ]:
# ============================================================
# Step 4G-2 -- FINAL QUALITY AUDIT
# ============================================================

final_quality_summary = pd.DataFrame({
    "Feature": CORE_FEATURES,
    "Non_Missing": [
        state_structural_panel[column].notna().sum()
        for column in CORE_FEATURES
    ],
    "Missing": [
        state_structural_panel[column].isna().sum()
        for column in CORE_FEATURES
    ],
    "Infinite": [
        np.isinf(state_structural_panel[column]).sum()
        for column in CORE_FEATURES
    ],
    "Minimum": [
        state_structural_panel[column].min()
        for column in CORE_FEATURES
    ],
    "Maximum": [
        state_structural_panel[column].max()
        for column in CORE_FEATURES
    ]
})

final_quality_summary["Coverage_Percent"] = (
    final_quality_summary["Non_Missing"]
    / len(state_structural_panel)
    * 100
)

display(final_quality_summary)

In [ ]:
## Structural validation 

assert len(state_structural_panel) == 500

assert (
    state_structural_panel["state_fips"]
    .nunique()
    == 50
)

assert (
    state_structural_panel["year"].min()
    == 2015
)

assert (
    state_structural_panel["year"].max()
    == 2024
)

assert not state_structural_panel.duplicated(
    ["state_fips", "year"]
).any()

assert (
    state_structural_panel[CORE_FEATURES]
    .notna()
    .all()
    .all()
)

assert np.isfinite(
    state_structural_panel[CORE_FEATURES]
).all().all()

print("Final structural-panel validation passed.")

## Step 1G-3 — Create chock-reponse features 

In [ ]:
# ============================================================
# CREATE ANNUAL GROWTH FEATURES
# ============================================================

growth_source_features = {
    "real_gdp_per_capita":
        "real_gdp_per_capita_growth",

    "real_personal_income_per_capita_2017":
        "real_income_per_capita_growth",

    "total_employment_jobs":
        "total_employment_growth",

    "real_average_wages_salaries_2017":
        "real_average_wage_growth"
}

dynamic_panel = (
    state_economic_clean[
        identifier_columns
        + list(growth_source_features.keys())
    ]
    .copy()
    .sort_values(["state_fips", "year"])
    .reset_index(drop=True)
)

for source_feature, growth_feature in growth_source_features.items():

    dynamic_panel[growth_feature] = (
        dynamic_panel
        .groupby("state_fips")[source_feature]
        .pct_change(fill_method=None)
        * 100
    )

GROWTH_FEATURES = list(
    growth_source_features.values()
)

display(
    dynamic_panel[
        identifier_columns + GROWTH_FEATURES
    ].head(12)
)

## Step 1G-4 — Create the three research periods

In [ ]:
# ============================================================
# ASSIGN RESEARCH PERIODS
# ============================================================

def assign_research_period(year):

    if 2015 <= year <= 2019:
        return "Baseline_2015_2019"

    if 2020 <= year <= 2022:
        return "Shock_2020_2022"

    if 2023 <= year <= 2024:
        return "Post_Shock_2023_2024"

    return np.nan


state_structural_panel["period"] = (
    state_structural_panel["year"]
    .apply(assign_research_period)
)

dynamic_panel["period"] = (
    dynamic_panel["year"]
    .apply(assign_research_period)
)

display(
    state_structural_panel[
        ["year", "period"]
    ]
    .drop_duplicates()
    .sort_values("year")
)

## Step 1G-5 — Create period-level clustering data

In [ ]:
# ============================================================
# PERIOD-LEVEL STRUCTURAL DATA
# ============================================================

state_period_structural = (
    state_structural_panel
    .groupby(
        ["state_fips", "state", "period"],
        as_index=False
    )[CORE_FEATURES]
    .mean()
)

print(
    "Structural period dataset:",
    state_period_structural.shape
)

print(
    "States:",
    state_period_structural["state"].nunique()
)

print(
    "Periods:",
    state_period_structural["period"].nunique()
)

display(state_period_structural.head())

In [ ]:
## Create period-level growth summaries 

state_period_growth = (
    dynamic_panel
    .groupby(
        ["state_fips", "state", "period"],
        as_index=False
    )[GROWTH_FEATURES]
    .mean()
)

print(
    "Growth period dataset:",
    state_period_growth.shape
)

display(state_period_growth.head())

## Step 1G-6 — Save all final cleaned datasets

In [ ]:
# ============================================================
# SAVE FINAL CLEANED DATASETS
# ============================================================

PROCESSED_DIR = Path("data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

panel_path = (
    PROCESSED_DIR /
    "state_structural_panel_core_2015_2024.csv"
)

period_path = (
    PROCESSED_DIR /
    "state_structural_period_averages.csv"
)

growth_panel_path = (
    PROCESSED_DIR /
    "state_dynamic_growth_panel_2015_2024.csv"
)

growth_period_path = (
    PROCESSED_DIR /
    "state_dynamic_growth_period_averages.csv"
)

state_structural_panel.to_csv(
    panel_path,
    index=False
)

state_period_structural.to_csv(
    period_path,
    index=False
)

dynamic_panel.to_csv(
    growth_panel_path,
    index=False
)

state_period_growth.to_csv(
    growth_period_path,
    index=False
)

print(f"Saved: {panel_path}")
print(f"Saved: {period_path}")
print(f"Saved: {growth_panel_path}")
print(f"Saved: {growth_period_path}")

### Final data-preparation decision

The final project uses **eight** fully prepared structural features covering
**economic capacity**, **productivity**, **compensation**, and **industry-employment
composition**.

Four additional labor, education, and demographic variables were considered.
However, repeated external API failures increased collection complexity, and
the variables were not essential for achieving the project's primary
unsupervised-learning objectives.

Instead, four annual growth indicators were constructed from the existing
data to support the analysis of the 2020–2022 shock. The structural variables
will be used for PCA and clustering, while the growth variables will help
interpret economic changes within and between clusters.

The final panel contains 500 observations representing 50 states from
2015 through 2024. It is divided into baseline, shock, and post-shock periods.

## Overall workflow: 

We now have only six main modeling stages:\
1. Load the final processed datasets in a new notebook.
2. Perform concise EDA and correlation analysis.
3. Standardize the eight structural features.
4. Apply PCA and interpret the components.
5. Apply K-Means and hierarchical clustering.
6. Compare baseline, shock, and post-shock cluster stability.